In [10]:
import os
import sys
import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download, HfApi

# TODO: Add these imports
# HINT: import torch
# HINT: import torch.nn.functional as F
# HINT: from transformers import AutoModel, AutoTokenizer
# HINT: from huggingface_hub import snapshot_download, HfApi

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print("Working directory:", os.getcwd())
print("Python:", sys.version)

Working directory: /Users/amir/Documents/quera-mlops/homework_03/B/HW3_Student/HW3_A
Python: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:07:49) [Clang 20.1.8 ]


# Step 1: Download model from HuggingFace

Download `sentence-transformers/all-MiniLM-L6-v2` and save 6 files to `bundle/model/`:

| # | File |
|---|------|
| 1 | `config.json` |
| 2 | `tokenizer_config.json` |
| 3 | `tokenizer.json` |
| 4 | `vocab.txt` |
| 5 | `special_tokens_map.json` |
| 6 | `model.safetensors` |

Use `snapshot_download` from `huggingface_hub` or download manually with `AutoModel.save_pretrained()` + `AutoTokenizer.save_pretrained()`.

After downloading, save the git commit hash to `bundle/model/.commit`.

In [11]:
# TODO: Download the 6 model files to bundle/model/

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED = ["config.json", "tokenizer_config.json", "tokenizer.json",
            "vocab.txt", "special_tokens_map.json", "model.safetensors"]
snapshot_download(
    repo_id=MODEL_ID,
    revision="main",
    local_dir=str(BUNDLE_MODEL_DIR),
    allow_patterns=REQUIRED,
)
# Save commit hash
commit = HfApi().model_info(MODEL_ID, revision="main").sha
(BUNDLE_MODEL_DIR / ".commit").write_text(commit + "\n", encoding="utf-8")
print(f"Saved commit hash: {commit}")


# HINT: Use snapshot_download(repo_id=MODEL_ID, revision="main",
#        local_dir=str(BUNDLE_MODEL_DIR), allow_patterns=[...])
# HINT: Required files: config.json, tokenizer_config.json, tokenizer.json,
#        vocab.txt, special_tokens_map.json, model.safetensors

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

# Save commit hash
# HINT: from huggingface_hub import HfApi
# HINT: commit = HfApi().model_info(MODEL_ID, revision="main").sha
# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

# Verify all 6 files exist
REQUIRED = ["config.json", "tokenizer_config.json", "tokenizer.json",
            "vocab.txt", "special_tokens_map.json", "model.safetensors"]
for fname in REQUIRED:
    fpath = BUNDLE_MODEL_DIR / fname
    assert fpath.exists(), f"MISSING: {fname}"
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  [OK] {fname:35s} {size_mb:7.1f} MB")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Saved commit hash: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
  [OK] config.json                             0.0 MB
  [OK] tokenizer_config.json                   0.0 MB
  [OK] tokenizer.json                          0.4 MB
  [OK] vocab.txt                               0.2 MB
  [OK] special_tokens_map.json                 0.0 MB
  [OK] model.safetensors                      86.7 MB


# Step 2: Write metadata.json

Create `bundle/metadata.json` with accurate information about the bundle.
Required fields:
- `model_name`: the HuggingFace model ID
- `model_revision`: the git commit hash from the `.commit` file
- `embedding_dim`: 384
- `max_seq_len`: 256
- `framework_version`: the installed torch version
- `transformers_version`: the installed transformers version
- `built_by`: YOUR NAME
- `build_timestamp_utc`: current time in ISO 8601 UTC format

In [12]:
# TODO: Create bundle/metadata.json

# HINT: commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
# HINT: ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
# HINT: torch_version = torch.__version__
# HINT: transformers_version = importlib.metadata.version("transformers")

# --- YOUR CODE HERE ---
import importlib

commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
torch_version = torch.__version__
transformers_version = importlib.metadata.version("transformers")

metadata = {
    # TODO: fill in all fields
    "model_name": MODEL_ID,
    "model_revision": commit,
    "embedding_dim": 384,
    "max_seq_len": 256,
    "framework_version": torch.__version__,
    "transformers_version": transformers_version,
    "built_by": "Amir",
}

# --- END YOUR CODE ---

# Write to file
meta_path = Path("bundle/metadata.json")
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "model_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "embedding_dim": 384,
  "max_seq_len": 256,
  "framework_version": "2.10.0",
  "transformers_version": "5.12.1",
  "built_by": "Amir"
}


# Step 3: Write MANIFEST.json

Compute SHA-256 hash for every file under `bundle/` (except MANIFEST.json itself)
and write the manifest to `bundle/MANIFEST.json`.

The format must be:
```json
{
  "format_version": 1,
  "files": {
    "relative/path/to/file": "sha256hexdigest",
    ...
  }
}
```

Pro tip: you can peek at `scripts/gen_manifest.py` for reference.

In [13]:
# TODO: Hash every file in bundle/ and create MANIFEST.json

# HINT: def sha256(filepath):
# HINT:     h = hashlib.sha256()
# HINT:     with open(filepath, "rb") as f:
# HINT:         for chunk in iter(lambda: f.read(1 << 20), b""):
# HINT:             h.update(chunk)
# HINT:     return h.hexdigest()

# --- YOUR CODE HERE ---

def sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Build and write manifest
root = Path("bundle")
files = {}
for p in sorted(root.rglob("*")):
    if p.is_file() and p.name != "MANIFEST.json":
        rel = str(p.relative_to(root))
        files[rel] = sha256(p)

manifest = {
    "format_version": 1,
    "files": files,
}

# Write
manifest_path = Path("bundle/MANIFEST.json")
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"MANIFEST.json written with {len(manifest['files'])} files")
for rel, h in sorted(manifest["files"].items()):
    print(f"  {h[:16]}...  {rel}")

MANIFEST.json written with 19 files
  b4d6d92b42602ebb...  __pycache__/predict.cpython-312.pyc
  120333e676eb1bac...  metadata.json
  684888c0ebb17f37...  model/.cache/huggingface/.gitignore
  9204359b30da80d5...  model/.cache/huggingface/download/config.json.metadata
  319a9fa3bbd22bdc...  model/.cache/huggingface/download/model.safetensors.metadata
  88c2e904de00e503...  model/.cache/huggingface/download/special_tokens_map.json.metadata
  fda229f0c922bb2b...  model/.cache/huggingface/download/tokenizer.json.metadata
  faf23de2d2903d2b...  model/.cache/huggingface/download/tokenizer_config.json.metadata
  77dbd76a3cf32a42...  model/.cache/huggingface/download/vocab.txt.metadata
  9d73db3316b1ee7a...  model/.commit
  e3b0c44298fc1c14...  model/.gitkeep
  953f9c0d463486b1...  model/config.json
  53aa51172d142c89...  model/model.safetensors
  303df45a03609e4e...  model/special_tokens_map.json
  be50c3628f2bf5bb...  model/tokenizer.json
  acb92769e8195aab...  model/tokenizer_config.json
 

# Step 4: Write predict.py

Write `bundle/predict.py` with exactly 4 functions:

| Function | Signature | Returns |
|----------|-----------|---------|
| `load_bundle()` | no args | `(model, tokenizer)` tuple |
| `embed(texts)` | `List[str]` | `np.ndarray` shape `(N, 384)` float32 |
| `similarity(a, b)` | two `np.ndarray` | `float` (cosine similarity) |
| `info()` | no args | `dict` with metadata |

The **7-step pipeline** inside `embed()`:
1. Tokenize: `tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")`
2. Move tensors to device (cpu or cuda)
3. Forward pass under `torch.no_grad()` → `last_hidden_state`
4. Mean-pool: `sum(H * mask) / sum(mask).clamp(min=1e-9)`
5. L2 normalize: `F.normalize(pooled, p=2, dim=1)`
6. Detach, move to CPU, convert to `np.float32`
7. Return ndarray

**Important rules:**
- DO NOT import `sentence-transformers`. Use raw `transformers` only.
- Set `torch.manual_seed(0)` for determinism.
- Call `model.eval()` before inference.
- A template already exists at `bundle/predict.py` — open it and fill in the TODOs.

In [14]:
# TODO: Write the implementation to bundle/predict.py
#
# Open bundle/predict.py in your editor and fill in the 4 functions:
#   load_bundle(), embed(), similarity(), info()
#
# Then run this cell to verify the module can be imported:

import sys
sys.path.insert(0, "bundle")

# HINT: After writing predict.py, uncomment these lines to test:
from predict import load_bundle, embed, similarity, info
model, tokenizer = load_bundle()
print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tokenizer).__name__)
print(info())

print("Edit bundle/predict.py, then re-run this cell to verify imports.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: BertModel
Tokenizer loaded: BertTokenizer
{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim': 384, 'max_seq_len': 256, 'device': 'cpu', 'framework': 'torch 2.10.0', 'deterministic': True, 'bundle_dir': '/Users/amir/Documents/quera-mlops/homework_03/B/HW3_Student/HW3_A/bundle/model'}
Edit bundle/predict.py, then re-run this cell to verify imports.


# Step 5: Run tests

Run all 4 test files. All tests must pass with green dots.

Tests check:
- **test_parity.py** (7 tests): Correct embedding shape, L2 normalization, similarity
- **test_tokenization.py** (5 tests): Tokenizer behavior, special tokens, round-trip
- **test_determinism.py** (1 test): Same input → same output every time
- **test_adversarial.py** (10 tests): Edge cases (unicode, numerics, long text, empty strings)

In [15]:
!pwd

/Users/amir/Documents/quera-mlops/homework_03/B/HW3_Student/HW3_A


In [16]:
# Run all tests
!PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

# Or if you're already in the HW3_A directory:
# !PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

============================= test session starts ==============================
platform darwin -- Python 3.12.12, pytest-9.1.1, pluggy-1.6.0 -- /Users/amir/miniconda3/envs/quera/bin/python
cachedir: .pytest_cache
rootdir: /Users/amir/Documents/quera-mlops/homework_03/B/HW3_Student/HW3_A
plugins: anyio-4.12.1, hydra-core-1.3.2
collected 23 items                                                             

tests/test_adversarial.py::test_missing_bundle_dir PASSED                [  4%]
tests/test_adversarial.py::test_very_long_text PASSED                    [  8%]
tests/test_adversarial.py::test_unicode_text PASSED                      [ 13%]
tests/test_adversarial.py::test_numeric_text PASSED                      [ 17%]
tests/test_adversarial.py::test_single_token_text PASSED                 [ 21%]
tests/test_adversarial.py::test_duplicate_texts PASSED                   [ 26%]
tests/test_adversarial.py::test_batch_of_one PASSED                      [ 30%]
tests/test_adversarial.py::te

# Step 6: Register in MLflow

Log the bundle to MLflow using `mlflow.sentence_transformers.log_model()`.

Before running this cell, make sure:
1. You have `source .env` (or set the env vars manually)
2. All tests pass
3. `bundle/model/` contains the 6 model files

In [17]:
from dotenv import load_dotenv

load_dotenv()


True

In [20]:
# TODO: Register the bundle in MLflow

# HINT: import mlflow
# HINT: mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
# HINT: mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))
#
# HINT: with mlflow.start_run(run_name="hw3a-bundle") as run:
# HINT:     mlflow.log_param("embedding_dim", 384)
# HINT:     mlflow.log_param("max_seq_len", 256)
# HINT:     mlflow.log_param("model_id", MODEL_ID)
# HINT:     mlflow.set_tag("stage", "candidate")
# HINT:
# HINT:     # Log the model
# HINT:     # Option A: sentence_transformers.log_model(...)
# HINT:     model_info = mlflow.sentence_transformers.log_model(
# HINT:         model_source=str(BUNDLE_MODEL_DIR.resolve()),
# HINT:         artifact_path="bundle",
# HINT:         task="llm/v1/embeddings",
# HINT:     )
# HINT:     print("Model URI:", model_info.model_uri)
# HINT:     print("Run ID:", run.info.run_id)

# --- YOUR CODE HERE ---
import mlflow
from sentence_transformers import SentenceTransformer
print(0)
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))
print(1)

with mlflow.start_run(run_name="hw3a-bundle") as run:
    print(2)
    mlflow.log_param("embedding_dim", 384)
    print(3)
    mlflow.log_param("max_seq_len", 256)
    mlflow.log_param("model_id", MODEL_ID)
    mlflow.set_tag("stage", "candidate")
    print("Here 1")

    st_model = SentenceTransformer(str(BUNDLE_MODEL_DIR.resolve()))
    model_info = mlflow.sentence_transformers.log_model(
        model=st_model,
        artifact_path="bundle",
        task="llm/v1/embeddings",
        registered_model_name=os.environ.get("MODEL_NAME"),
    )

    print("Model URI:", model_info.model_uri)
    print("Run ID:", run.info.run_id)
# --- END YOUR CODE ---

print("Done. Check MLflow UI for your registered model.")

0
1
2
3
Here 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully registered model 'hw03-modelv0.0.1'.
2026/06/26 21:23:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: hw03-modelv0.0.1, version 1
Created version '1' of model 'hw03-modelv0.0.1'.


Model URI: runs:/a79262e2cdc34436b151515f7c0abdf0/bundle
Run ID: a79262e2cdc34436b151515f7c0abdf0
🏃 View run hw3a-bundle at: http://185.50.38.163:33014/#/experiments/19/runs/a79262e2cdc34436b151515f7c0abdf0
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/19
Done. Check MLflow UI for your registered model.


# Step 7: Upload to MinIO

Upload your `bundle/` directory to the shared MinIO bucket.

Run the provided upload script:
```bash
source .env && bash scripts/01_upload_to_minio.sh
```

Or from this notebook:

In [ ]:
# Upload bundle to MinIO using the provided script
# Make sure .env is sourced first!

# in mac this works:
# set -a && source .env && set +a 
# bash scripts/01_upload_to_minio.sh

!bash scripts/01_upload_to_minio.sh

# Take a screenshot of the upload confirmation for EVIDENCE/minio_upload.png

## Submission Checklist

Before submitting, verify:

- [ ] `encoder_bundle.ipynb` — all cells executed with visible outputs
- [ ] `bundle/predict.py` — 4 functions implemented
- [ ] `bundle/metadata.json` — all fields filled (no TODO placeholders)
- [ ] `bundle/MANIFEST.json` — real SHA-256 hashes for every file
- [ ] `bundle/requirements.txt` — pinned dependencies listed
- [ ] `bundle/model/` — 6 model files present
- [ ] `EVIDENCE/pytest_pass.png` — all tests green
- [ ] `EVIDENCE/mlflow_registered.png` — MLflow UI showing the model
- [ ] `EVIDENCE/minio_upload.png` — MinIO upload confirmation

Good luck!